In [ ]:
# importing necessary library
import os
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy

In [ ]:
# Installing Hugging Face datasets library
%pip install datasets

In [ ]:
# Loading the arXiv dataset with article and abstract
from datasets import load_dataset
dataset = load_dataset("scientific_papers", "arxiv", split="train[:1%]")

In [ ]:
# Checking structure of the dataset
print(dataset[0].keys())

In [ ]:
# Training Byte-Level BPE tokenizer from scratch
from tokenizers import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing
# Creating corpus file from article and abstract
with open("corpus.txt", "w", encoding="utf-8") as f:
    for sample in dataset:
        f.write(sample['article'].replace("\n", " ") + "\n")   # input
        f.write(sample['abstract'].replace("\n", " ") + "\n")  # target/summary
# Initializing and training tokenizer
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files="corpus.txt",
    vocab_size=30522,
    min_frequency=2,
    special_tokens=["<s>", "</s>", "<pad>", "<unk>"]
)
import os
os.makedirs("custom_tokenizer", exist_ok=True)
tokenizer.save_model("custom_tokenizer")
# Loading the trained tokenizer
tokenizer = ByteLevelBPETokenizer(
    "custom_tokenizer/vocab.json",
    "custom_tokenizer/merges.txt"
)
tokenizer._tokenizer.post_processor = BertProcessing(
    ("</s>", tokenizer.token_to_id("</s>")),
    ("<s>", tokenizer.token_to_id("<s>"))
)
tokenizer.enable_truncation(max_length=128)

In [ ]:
# Plotting token length distribution of articles and summaries
import matplotlib.pyplot as plt
article_lengths = [len(tokenizer.encode(sample["article"]).ids) for sample in dataset]
summary_lengths = [len(tokenizer.encode(sample["abstract"]).ids) for sample in dataset]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(article_lengths, bins=30, color="skyblue")
plt.title("Article Token Length Distribution")
plt.xlabel("Tokens")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.hist(summary_lengths, bins=30, color="lightgreen")
plt.title("Summary Token Length Distribution")
plt.xlabel("Tokens")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
# Defining preprocessing to tokenize and pad inputs/targets
def preprocess(example):
    input_ids = tokenizer.encode(example["article"]).ids[:128]
    target_ids = tokenizer.encode(example["abstract"]).ids[:32]

    input_ids += [tokenizer.token_to_id("<pad>")] * (128 - len(input_ids))
    target_ids = [tokenizer.token_to_id("<s>")] + target_ids + [tokenizer.token_to_id("</s>")]
    target_ids += [tokenizer.token_to_id("<pad>")] * (34 - len(target_ids))

    return {"input_ids": input_ids, "labels": target_ids}


In [ ]:
# Applying preprocessing function to the dataset
dataset = dataset.map(preprocess)

In [ ]:
# Checking keys of processed dataset
print(dataset[0].keys())

In [ ]:
# Defining helper to clone a module to be used in Transformer
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [ ]:
# Defining Positional Encoding for token embeddings
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Creating constant 'pe' matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Adding position info to input embeddings
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
# Creating Layer Normalization
class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps
    # Applying normalization
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [ ]:
# Creating Sublayer connection block with residual + normalization
class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
    # Applying residual connection and normalization
    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

In [ ]:
# Creating Scaled Dot-Product Attention
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)# getting dimension of key/query
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)# calculating scaled dot-product
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)# applying mask to ignore padding
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)# applying dropout to attention weights
    return torch.matmul(p_attn, value), p_attn

In [ ]:
# Creating Multi-Head Attention module
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0# making sure model dimension is divisible by number of heads
        self.d_k = d_model // h# calculating dimension per head
        self.h = h# storing number of heads
        # creating linear layers for query, key, value, and output projection
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)# setting dropout

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)# adjusting mask shape for multi-head
        nbatches = query.size(0)# getting batch size
        # projecting query, key, and value for each head and reshaping
        query, key, value = [
            l(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for l, x in zip(self.linears, (query, key, value))
        ]
        # computing attention and getting weighted values
        x, self.attn = attention(query, key, value, mask, self.dropout)
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.h * self.d_k)# reshaping back and combining heads
        return self.linears[-1](x)

In [ ]:
# Creating Feedforward layer used in Transformer
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)# projecting input to higher dimension
        self.w_2 = nn.Linear(d_ff, d_model)# projecting back to original dimension
        self.dropout = nn.Dropout(dropout)# applying dropout between layers
    # Applying two-layer feedforward with ReLU and dropout
    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))

In [ ]:
# Creating a single Encoder layer self-attention + feedforward
class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn# setting self-attention layer
        self.feed_forward = feed_forward# setting feedforward layer
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size
    # Applying self-attention followed by feedforward, with residual connections
    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

In [ ]:
# Creating a single Decoder layer: masked self-attn, encoder-decoder attn, feedforward
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn# setting masked self-attention
        self.src_attn = src_attn# setting encoder-decoder attention
        self.feed_forward = feed_forward# setting feedforward layer
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory# storing encoder output as memory
        # Applying masked self-attention, encoder-decoder attention, then feedforward
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

In [ ]:
# Creating Encoder stack by stacking multiple Encoder layers
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)# cloning the EncoderLayer N times
        self.norm = LayerNorm(layer.size)# applying final normalization

    def forward(self, x, mask):
      # Passing input through each Encoder layer
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)# returning final normalized output
# Creating Decoder stack by stacking multiple Decoder layers
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
      # Passing input through each Decoder layer
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
            # returning final normalized output
        return self.norm(x)

In [ ]:
# Creating token embeddings with positional scaling
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)# looking up embedding table
        self.d_model = d_model

    def forward(self, x):
      # Scaling embeddings by sqrt(d_model) to stabilize gradients
        return self.lut(x) * math.sqrt(self.d_model)

In [ ]:
# Creating the full Transformer model: Encoder + Decoder + Generator
class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder# setting encoder stack
        self.decoder = decoder# setting decoder stack
        self.src_embed = src_embed# embedding + positional encoding for source
        self.tgt_embed = tgt_embed# embedding + positional encoding for target
        self.generator = generator# projection to vocabulary + softmax

    def forward(self, src, tgt, src_mask, tgt_mask):
      # Encoding source, then decoding target based on encoded memory
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
      # Embedding and passing source into encoder
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
      # Embedding and decoding target using encoder memory
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [ ]:
# Creating the Generator to convert decoder output to vocabulary prediction
class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
      # Applying log softmax to get token probabilities
        return F.log_softmax(self.proj(x), dim=-1)

# Creating a mask to block attention to future positions in the decoder
def subsequent_mask(size):
    attn_shape = (1, size, size)
    mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    return mask == 0

In [ ]:
# Building the full Transformer model
def make_model(vocab_size, N=6, d_model=768, d_ff=3072, h=8, dropout=0.2):
  # Creating attention, feedforward, and positional encoding layers
    attn = MultiHeadedAttention(h, d_model)# setting multi-head attention
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)# creating feedforward network
    position = PositionalEncoding(d_model, dropout)# applying positional encoding
    # Constructing the full Encoder-Decoder architecture
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, attn, ff, dropout), N),# stacking N encoder layers
        Decoder(DecoderLayer(d_model, attn, attn, ff, dropout), N),# stacking N decoder layers
        nn.Sequential(Embeddings(d_model, vocab_size), position),# embedding + positional encoding for source
        nn.Sequential(Embeddings(d_model, vocab_size), position),# embedding + positional encoding for target
        Generator(d_model, vocab_size)# output generator projecting to vocab size
    )
    return model

In [ ]:
# Creating mask to block future tokens in decoder during training
def subsequent_mask(size):
    # Masking out upper triangular matrix
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    return subsequent_mask == 0

In [ ]:
# Importing utilities for dataset handling
from torch.utils.data import Dataset, DataLoader
# Wrapping tokenized HuggingFace dataset into PyTorch Dataset
class ArxivDataset(Dataset):
    def __init__(self, data):
        self.data = data# storing preprocessed dataset

    def __len__(self):
        return len(self.data)# returning dataset length

    def __getitem__(self, idx):
      # Getting tokenized input-output pair
        item = self.data[idx]
        src = torch.tensor(item["input_ids"])# getting source tokens
        tgt = torch.tensor(item["labels"])# getting target summary tokens
        return src, tgt

# Creating DataLoader to load batches during training
dataset = ArxivDataset(dataset)  # converting HuggingFace dataset to PyTorch Dataset
train_loader = DataLoader(dataset, batch_size=8, shuffle=True)# loading data in batches with shuffling

In [ ]:
# Function to generate summary from input text using greedy decoding
def generate_summary(model, input_text, tokenizer, max_len=64):
    model.eval()# setting model to evaluation mode

    # Tokenizing the input article
    input_ids = tokenizer.encode(input_text).ids[:128]# trimming to max 128 tokens
    input_ids += [tokenizer.token_to_id("<pad>")] * (128 - len(input_ids))# padding to fixed length
    src = torch.tensor([input_ids]).to(next(model.parameters()).device)# creating batch and moving to device
    src_mask = (src != tokenizer.token_to_id("<pad>")).unsqueeze(-2)# creating source mask

    # Setting start and end token IDs
    start_symbol = tokenizer.token_to_id("<s>")
    eos_symbol = tokenizer.token_to_id("</s>")

    # Decoding output tokens using greedy strategy
    decoded_ids = greedy_decode(model, src, src_mask, max_len, start_symbol, eos_symbol)

    # Removing special tokens and decoding to text
    token_ids = [t for t in decoded_ids[0].tolist() if t not in [
    tokenizer.token_to_id("<pad>"),
    tokenizer.token_to_id("<s>"),
    tokenizer.token_to_id("</s>")]]

    return tokenizer.decode(token_ids)

In [ ]:
# Loading original raw dataset for evaluation
from datasets import load_dataset
raw_dataset = load_dataset("scientific_papers", "arxiv", split="train[:1%]")

In [ ]:
# Downloading NLTK resources and installing metrics libraries
import nltk
nltk.download('punkt_tab')

In [ ]:
%pip install rouge_score
%pip install bert_score

In [ ]:
# Importing necessary libraries for training and evaluation
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore
from nltk.tokenize import word_tokenize

# Setting the device, GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initializing model, loss, and optimizer
vocab_size = tokenizer.get_vocab_size()
pad_token_id = tokenizer.token_to_id("<pad>")
model = make_model(vocab_size).to(device)# creating and sending model to device
loss_fn = nn.NLLLoss(ignore_index=pad_token_id)# ignoring pad token in loss
optimizer = optim.Adam(model.parameters(), lr=1e-4)# setting learning rate

# Initializing lists to track metrics
epochs = 250
loss_history = []
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []
bleu_scores = []
bertscore_f1s = []

# Setting up metric helpers
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smoothie = SmoothingFunction().method4# smoothing for BLEU

# Starting training loop
for epoch in range(epochs):
    model.train()# setting model to training mode
    total_loss = 0
    print(f"\nEpoch {epoch+1}/{epochs}")
    # Iterating through batches
    for src, tgt in tqdm(train_loader):
        src = src.to(device)
        tgt = tgt.to(device)
        # Preparing decoder input and output
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        # Creating masks for source and target
        src_mask = (src != pad_token_id).unsqueeze(-2)
        tgt_mask = (tgt_input != pad_token_id).unsqueeze(-2) & subsequent_mask(tgt_input.size(-1)).to(device)
        # Running forward pass
        out = model(src, tgt_input, src_mask, tgt_mask)
        logits = model.generator(out)
        # Reshaping tensors for loss calculation
        logits = logits.view(-1, logits.size(-1))
        tgt_output = tgt_output.contiguous().view(-1)
        # Calculating and updating loss
        loss = loss_fn(logits, tgt_output)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Calculating average loss for the epoch
    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Avg Loss: {avg_loss:.4f}")

    # Evaluate on 1 sample
    model.eval()
    sample = raw_dataset[0]["article"]
    reference = raw_dataset[0]["abstract"]
    prediction = generate_summary(model, sample, tokenizer)

    # Calculating ROUGE scores
    scores = scorer.score(reference, prediction)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

    # Calculating BLEU score
    ref_tokens = word_tokenize(reference)
    pred_tokens = word_tokenize(prediction)
    bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie)
    bleu_scores.append(bleu)

    # Calculating BERTScore
    _, _, f1 = bertscore([prediction], [reference], lang="en", verbose=False)
    bertscore_f1s.append(f1[0].item())

    # Printing all evaluation metrics
    print(f"ROUGE-1: {scores['rouge1'].fmeasure:.4f} | ROUGE-2: {scores['rouge2'].fmeasure:.4f} | ROUGE-L: {scores['rougeL'].fmeasure:.4f}")
    print(f" BLEU: {bleu:.4f} |  BERTScore F1: {f1[0].item():.4f}")

In [ ]:
# Plotting training metrics over epochs
plt.figure(figsize=(14, 8))

# Plotting loss per epoch
plt.subplot(2, 3, 1)
plt.plot(loss_history, marker='o')
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
# Plotting ROUGE scores per epoch
plt.subplot(2, 3, 2)
plt.plot(rouge1_scores, label="ROUGE-1", marker='o')
plt.plot(rouge2_scores, label="ROUGE-2", marker='s')
plt.plot(rougeL_scores, label="ROUGE-L", marker='^')
plt.title("ROUGE Scores")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.legend()
plt.grid(True)
# Plotting BLEU score per epoch
plt.subplot(2, 3, 3)
plt.plot(bleu_scores, marker='*', color='orange')
plt.title("BLEU Score")
plt.xlabel("Epoch")
plt.ylabel("BLEU")
plt.grid(True)
# Plotting BERTScore F1 per epoch
plt.subplot(2, 3, 4)
plt.plot(bertscore_f1s, marker='d', color='purple')
plt.title("BERTScore F1")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Defining decoding function for summary generation
def greedy_decode(model, src, src_mask, max_len, start_symbol, eos_symbol):
    model.eval()# setting model to eval mode
    memory = model.encode(src, src_mask)

    # Initializing decoder with <s> token
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src).to(src.device)

    for _ in range(max_len - 1):
        tgt_mask = subsequent_mask(ys.size(1)).to(src.device).unsqueeze(1)# creating mask for decoding
        out = model.decode(memory, src_mask, ys, tgt_mask)# decoding step

        # Sampling next token using probabilities instead of argmax
        probs = torch.softmax(model.generator(out[:, -1]), dim=-1)
        next_word = torch.multinomial(probs, num_samples=1)
        # appending predicted token
        ys = torch.cat([ys, next_word], dim=1)

        # stopping if </s> is predicted
        if next_word.item() == eos_symbol:
            break

    return ys


In [ ]:
# Creating a cleaning function to preprocess raw data
import re

def clean_and_filter(example):
    # Replacing @xmath variables with [MATH] placeholder
    article = re.sub(r"@xmath\d+", "[MATH]", example["article"])
    abstract = re.sub(r"@xmath\d+", "[MATH]", example["abstract"])

    # Filtering articles with acceptable character length
    if 50 <= len(article) <= 1000:
        return {
            "article": article,
            "abstract": abstract,
            "section_names": example.get("section_names", "")
        }
    else:
        return None  # dropping article if it's too short/long


In [ ]:
# Running and printing predictions for a few samples
for i in range(5):
    print(f"\nSample {i+1}")
    article = raw_dataset[i]["article"]# getting source text
    target = raw_dataset[i]["abstract"]# getting reference summary
    predicted = generate_summary(model, article, tokenizer)# generating prediction

    print(" Reference:", target[:200], "...")# printing first 200 characters of reference
    print(" Generated:", predicted)# printing generated summary
    print("-" * 80)